# Notebook 17 v2 — SD3 FlowEdit neutral-target pilot (resume-safe)

**EXECUTED 2026-10-14 — completed 24/24; requires Colab GPU and Hugging Face gated access.** Recovery-oriented replacement for N17 v1. The 24 prior jobs returned code 0 but the outputs were saved only under ephemeral `/content/flowedit_nb17/outputs`, rather than copied to Drive. V2 starts a new, clearly identified run and **persists each PNG immediately after each job**, with integrity checks and durable receipts. It can resume safely after runtime interruption. Does not claim measured privacy or emotion performance.

Pilot: 12 distinct held-out AffectNet sources × seeds 42 and 123, neutral-expression target prompt, guidance 5.5, original frozen N07 source prompts. Reuses the original FlowEdit **commit** and explicitly pins the SD3 **revision** inside its runner. Keep outputs private; public repository may contain only clean code and aggregate results.

**Important:** It is an *expression-target ablation*, not proof of anonymization. Do not compare model outputs until evaluation has been run. The notebook reuses historical revision/config; model compatibility and resulting image meaning still require empirical checking.

In [ ]:
# 1 — Mount, deterministic design, persistent output (idempotent on re-run)
from google.colab import drive, userdata
drive.mount('/content/drive')
from pathlib import Path
import os,sys,hashlib,json,subprocess,time,shutil,re
import numpy as np,pandas as pd,yaml
from datetime import datetime,timezone
from PIL import Image

ROOT=Path('/content/drive/MyDrive/thesis_all/outputs/runs')
N07=ROOT/'20260818T210720Z_notebook07_affectnet_larger_6aeb3643_f1142664'
FLOW_COMMIT='85594626161caca9dafa68592c1ffa21db87a806'
SD3_REV='ea42f8cef0f178587cf766dc8129abd379c90671'
MODEL='stabilityai/stable-diffusion-3-medium-diffusers'
RUN_TAG='nb17v2_neutral_expression_12sources_2seeds_sd3pinned'
OUT=ROOT/RUN_TAG   # fixed name: supports resuming in subsequent Colab runtimes
for dirname in ['images','logs','configs','staging']:(OUT/dirname).mkdir(parents=True,exist_ok=True)
assert (N07/'selected_sources.csv').is_file()
sources=pd.read_csv(N07/'selected_sources.csv')
tuning={'source_000','source_001','source_005','source_006','source_010','source_011','source_015','source_016','source_020','source_021','source_025','source_026'}
assert len(sources)==120 and sources.source_id.is_unique and sources.emotion.nunique()==6
eligible=sources.loc[~sources.source_id.isin(tuning)].copy()
pilot=pd.concat([g.sort_values('source_id').head(2) for _,g in eligible.groupby('emotion',sort=True)],ignore_index=True)
assert len(pilot)==12 and pilot.source_id.nunique()==12
pilot[['source_id','emotion']].to_csv(OUT/'pilot_sources_PRIVATE.csv',index=False)
print('Persistent output:',OUT,'\nSources:',pilot[['source_id','emotion']].to_string(index=False))

In [ ]:
# 2 — Reuse EXACT N07 per-source source prompt and define neutral-expression target
cfgs=[]
for r in pilot.itertuples(index=False):
    sid=r.source_id
    evidence=N07/'flowedit_configs'/f'nb07_{sid}_flowedit_g5p5_s42_f1142664_edits.yaml'
    assert evidence.is_file(),evidence
    d=yaml.safe_load(evidence.read_text())
    assert isinstance(d,list) and len(d)==1
    source_prompt=d[0]['source_prompt']; target_previous=d[0]['target_prompts'][0]
    assert str(r.emotion) in source_prompt and 'different person' in target_previous.lower()
    target='A realistic close-up portrait photograph of a different person with a neutral, relaxed facial expression, keeping similar head angle and eye gaze to the original. Photorealistic.'
    source=N07/'sources'/f'{sid}.png'
    assert source.is_file()
    expected=str(sources.set_index('source_id').loc[sid,'prepared_source_sha256'])
    assert hashlib.sha256(source.read_bytes()).hexdigest()==expected
    cfgs.append({'source_id':sid,'emotion':r.emotion,'source_prompt':source_prompt,'previous_target_prompt':target_previous,'neutral_target_prompt':target,'source_sha256':expected})
manifest=OUT/'prompt_manifest_PRIVATE.json'
serialized=json.dumps(cfgs,indent=2)
if manifest.exists(): assert json.loads(manifest.read_text())==cfgs,'Prompt manifest changed across resume; STOP'
else: manifest.write_text(serialized)
print('Prompt and source SHA manifest verified for',len(cfgs),'sources')

In [ ]:
# 3 — Environment, credential + EXACT model revision and FlowEdit commit
import torch
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > T4 GPU or better'
subprocess.run([sys.executable,'-m','pip','-q','install','diffusers==0.30.1','transformers==4.44.2','accelerate','sentencepiece','protobuf','safetensors','huggingface_hub==0.36.2'],check=True)
from huggingface_hub import HfApi,login
HF_TOKEN=userdata.get('HF_TOKEN')
assert HF_TOKEN,'Set HF_TOKEN in Colab Secrets and enable notebook access'
login(token=HF_TOKEN,add_to_git_credential=False)
info=HfApi().model_info(repo_id=MODEL,revision=SD3_REV,token=HF_TOKEN)
assert info.sha==SD3_REV
WORK=Path('/content/flowedit_nb17v2')
if not WORK.exists():subprocess.run(['git','clone','-q','https://github.com/fallenshock/FlowEdit.git',str(WORK)],check=True)
subprocess.run(['git','-C',str(WORK),'checkout','--quiet',FLOW_COMMIT],check=True)
actual_commit=subprocess.check_output(['git','-C',str(WORK),'rev-parse','HEAD'],text=True).strip()
assert actual_commit==FLOW_COMMIT
runner=WORK/'run_script.py'
original=subprocess.check_output(['git','-C',str(WORK),'show',f'{FLOW_COMMIT}:run_script.py'],text=True)
needle='StableDiffusion3Pipeline.from_pretrained("stabilityai/stable-diffusion-3-medium-diffusers", torch_dtype=torch.float16)'
replacement='StableDiffusion3Pipeline.from_pretrained("stabilityai/stable-diffusion-3-medium-diffusers", revision="'+SD3_REV+'", torch_dtype=torch.float16)'
assert original.count(needle)==1,'Unexpected upstream runner, STOP'
patched=original.replace(needle,replacement)
assert patched.count('revision="'+SD3_REV+'"')==1
runner.write_text(patched)
assert replacement in runner.read_text()
receipt={'flowedit_commit':actual_commit,'sd3_model':MODEL,'sd3_revision':SD3_REV,'runner_original_sha256':hashlib.sha256(original.encode()).hexdigest(),'runner_patched_sha256':hashlib.sha256(patched.encode()).hexdigest(),'python':sys.version,'torch':torch.__version__,'cuda_device':torch.cuda.get_device_name(0)}
(OUT/'environment_receipt.json').write_text(json.dumps(receipt,indent=2))
print('Verified FlowEdit commit:',actual_commit,'\nPinned SD3 revision:',SD3_REV,'\nGPU:',receipt['cuda_device'])

In [ ]:
# 4 — Crash-safe generation: each job copied to Drive immediately; skip verified outputs
# Resume after interruption by running cells 1-4 again; never run multiple instances concurrently.
JOBS=[(c,s) for c in cfgs for s in (42,123)]
assert len(JOBS)==24
(WORK/'example_images').mkdir(exist_ok=True)

def sha256(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def valid_png(path):
    try:
        with Image.open(path) as im:im.verify()
        with Image.open(path) as im:
            assert im.size[0]>=256 and im.size[1]>=256
            return im.size
    except Exception:return None

def persist_receipts(records):
    tmp=OUT/'receipts_tmp.csv';pd.DataFrame(records).to_csv(tmp,index=False);os.replace(tmp,OUT/'private_execution_receipts.csv')

def output_for_run(name):
    # Upstream pinned runner code saves: outputs/{exp_name}/SD3/src_{input_stem}/tar_{tar_num}/output_*.png
    root=WORK/'outputs'/name/'SD3'/('src_'+name)/'tar_0'
    candidates=list(root.glob('output_*.png')) if root.is_dir() else []
    assert len(candidates)==1,f'Expected one output in {root}, got {len(candidates)}; inspect private logs.'
    return candidates[0]

records=[]
for index,(c,seed) in enumerate(JOBS,1):
    sid=c['source_id'];name=f'nb17v2_{sid}_neutral_g5p5_s{seed}'
    final=OUT/'images'/f'{name}.png'
    previous=OUT/'images'/f'{name}.receipt.json'
    if final.exists() and previous.exists():
        saved=json.loads(previous.read_text())
        assert saved['output_sha256']==sha256(final) and valid_png(final),f'Existing output corrupt: {final}'
        assert saved['source_sha256']==c['source_sha256'] and saved['sd3_revision']==SD3_REV
        assert saved['runner_commit']==FLOW_COMMIT and saved['seed']==seed
        records.append(saved);persist_receipts(records)
        print(f'[{index}/24] RESUMED {name}',flush=True);continue
    if final.exists() or previous.exists():raise RuntimeError(f'Partial persisted job {name}; STOP for inspection (do not overwrite).')
    original_img=N07/'sources'/f'{sid}.png'
    assert sha256(original_img)==c['source_sha256']
    stage=WORK/'example_images'/f'{name}.png';shutil.copy2(original_img,stage)
    edit_path=OUT/'configs'/f'{name}_edits.yaml'
    exp_path=OUT/'configs'/f'{name}_exp.yaml'
    ed=[{'input_img':f'example_images/{name}.png','source_prompt':c['source_prompt'],'target_prompts':[c['neutral_target_prompt']],'target_codes':['neutral_expression_different_person']}]
    exp=[{'exp_name':name,'dataset_yaml':str(edit_path),'model_type':'SD3','sampler_type':'FlowEditSD3','T_steps':40,'src_guidance_scale':3.5,'tar_guidance_scale':5.5,'n_min':0,'n_max':33,'n_avg':1,'seed':int(seed)}]
    edit_path.write_text(yaml.safe_dump(ed,sort_keys=False));exp_path.write_text(yaml.safe_dump(exp,sort_keys=False))
    logfile=OUT/'logs'/f'{name}.txt';start=time.time()
    env=os.environ.copy();env['HF_TOKEN']=HF_TOKEN
    print(f'[{index}/24] GENERATING {name}',flush=True)
    with logfile.open('w') as stream:
        job=subprocess.run([sys.executable,'run_script.py','--exp_yaml',str(exp_path)],cwd=WORK,env=env,stdout=stream,stderr=subprocess.STDOUT)
    if job.returncode!=0:raise RuntimeError(f'{name} failed with exit code {job.returncode}; see {logfile}')
    generated=output_for_run(name)
    dims=valid_png(generated)
    assert dims is not None,f'Output image invalid: {generated}'
    temp=OUT/'staging'/f'{name}.png.tmp'
    shutil.copyfile(generated,temp)
    assert valid_png(temp)==dims
    os.replace(temp,final)
    saved={'source_id':sid,'emotion':c['emotion'],'seed':int(seed),'run_name':name,'status':'VERIFIED','source_sha256':c['source_sha256'],'output_path':str(final),'output_sha256':sha256(final),'image_size':str(dims),'runner_commit':FLOW_COMMIT,'sd3_revision':SD3_REV,'duration_seconds':round(time.time()-start,2),'runner_log':str(logfile),'runner_output':str(generated)}
    previous.write_text(json.dumps(saved,indent=2))
    records.append(saved);persist_receipts(records)
    print(f'[{index}/24] SAVED AND VERIFIED {final.name}',flush=True)
print('DONE:',len(records),'verified images stored persistently in Drive')

In [ ]:
# 5 — Independent persistent postflight; no dependence on in-memory records
receipts_dir=OUT/'images'
all_receipts=sorted(receipts_dir.glob('*.receipt.json'))
expected={f'nb17v2_{c["source_id"]}_neutral_g5p5_s{seed}' for c in cfgs for seed in (42,123)}
seen=set();valid=0
for recfile in all_receipts:
    r=json.loads(recfile.read_text());name=r['run_name']
    assert name in expected and name not in seen;seen.add(name)
    path=receipts_dir/f'{name}.png'
    assert path.is_file() and valid_png(path) and sha256(path)==r['output_sha256']
    assert r['runner_commit']==FLOW_COMMIT and r['sd3_revision']==SD3_REV
    valid+=1
status={'status':'COMPLETE' if seen==expected else 'PARTIAL','expected_jobs':24,'verified_images':valid,'remaining':len(expected-seen),'flowedit_commit':FLOW_COMMIT,'sd3_revision_requested_and_pinned':SD3_REV,'emotion_evaluation':'NOT_RUN','biometric_evaluation':'NOT_RUN','pilot_type':'neutral_expression_prompt_ablation'}
(OUT/'status.json').write_text(json.dumps(status,indent=2))
print(json.dumps(status,indent=2))
assert seen==expected, f'Missing: {sorted(expected-seen)}'
print('All 24 image files saved and hash-verified on Drive. Evaluation remains separate.')